# 🎯 Evaluate All Segmentation Models

This notebook evaluates and compares all trained segmentation models on the test/validation dataset.

**Models to Compare:**
- UNet
- AttentionUNet
- ResUNetPP
- SwinUNet

**Metrics:**
- Dice Coefficient
- IoU Score
- Sensitivity
- Specificity
- Precision

## 1. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Import Modules

In [ ]:
from src.segmentation import TumorSegmentor
from src.segmentation.data import create_segmentation_datasets
from src.segmentation.training.metrics import (
    dice_coefficient,
    iou_score,
    sensitivity,
    specificity,
    precision_metric
)

print("✅ Modules imported successfully!")

## 3. Configuration

In [ ]:
# Data paths
TEST_IMAGES_DIR = "../data/brisc2025/segmentation_task/test/images"
TEST_MASKS_DIR = "../data/brisc2025/segmentation_task/test/masks"

# Models to evaluate (add your trained models here)
MODELS = {
    'UNet': '../weights/segmentation/UNet_bce_tversky_best.keras',
    'AttentionUNet': '../weights/segmentation/AttentionUNet_bce_tversky_best.keras',
    'ResUNetPP': '../weights/segmentation/ResUNetPP_bce_tversky_best.keras',
    'SwinUNet': '../weights/segmentation/SwinUNet_bce_tversky_best.keras',
}

# Filter only existing models
AVAILABLE_MODELS = {name: path for name, path in MODELS.items() if os.path.exists(path)}

print(f"\n📊 Models to evaluate: {list(AVAILABLE_MODELS.keys())}")
print(f"   Total: {len(AVAILABLE_MODELS)} models")

# Settings
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
THRESHOLD = 0.5

# Output directory
OUTPUT_DIR = "../logs/segmentation/evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ Configuration complete!")
print(f"   Output directory: {OUTPUT_DIR}")

## 4. Prepare Test Dataset

In [ ]:
# Create dataset
train_ds, val_ds, _ = create_segmentation_datasets(
    train_images_dir=TEST_IMAGES_DIR,
    train_masks_dir=TEST_MASKS_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=0.2,
    random_state=42,
    use_augmentation=False,  # No augmentation for testing
)

# Use validation set for evaluation
test_ds = val_ds

print(f"✅ Test dataset prepared")

## 5. Evaluate All Models

In [ ]:
results = {}

for model_name, model_path in AVAILABLE_MODELS.items():
    print(f"\n{'='*60}")
    print(f"📊 Evaluating {model_name}...")
    print(f"   Model: {model_path}")
    print(f"{'='*60}")
    
    # Load model
    model = tf.keras.models.load_model(model_path)
    
    # Evaluate
    eval_results = model.evaluate(test_ds, verbose=1)
    
    # Extract metrics
    metric_names = ["Loss"] + [m.name for m in model.metrics]
    results[model_name] = dict(zip(metric_names[:len(eval_results)], eval_results))
    
    print(f"\n✅ {model_name} evaluation complete!")
    for metric, value in results[model_name].items():
        print(f"   {metric:.<30} {value:.4f}")

print(f"\n{'='*60}")
print("✅ All models evaluated successfully!")
print(f"{'='*60}")

## 6. Create Results DataFrame

In [ ]:
# Convert to DataFrame
results_df = pd.DataFrame(results).T

# Round values
results_df = results_df.round(4)

# Save to CSV
csv_path = f"{OUTPUT_DIR}/all_models_comparison.csv"
results_df.to_csv(csv_path)
print(f"💾 Results saved to: {csv_path}")

# Display
print("\n📊 Model Comparison:")
display(results_df)

# Find best model for each metric
print("\n🏆 Best Models per Metric:")
for metric in results_df.columns:
    if metric.lower() == 'loss':
        best_model = results_df[metric].idxmin()
        best_value = results_df[metric].min()
    else:
        best_model = results_df[metric].idxmax()
        best_value = results_df[metric].max()
    print(f"   {metric:.<30} {best_model} ({best_value:.4f})")

## 7. Visualizations

### 7.1 Bar Chart with Error Bars

In [ ]:
# Select key metrics (excluding loss)
key_metrics = [m for m in results_df.columns if m.lower() != 'loss']

# Create bar chart
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(key_metrics))
width = 0.15
colors = plt.cm.tab10(np.linspace(0, 1, len(AVAILABLE_MODELS)))

for i, (model_name, color) in enumerate(zip(results_df.index, colors)):
    offset = width * (i - len(results_df.index) / 2 + 0.5)
    values = [results_df.loc[model_name, m] for m in key_metrics]
    ax.bar(x + offset, values, width, label=model_name, color=color, alpha=0.8)

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Segmentation Models Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(key_metrics, rotation=0)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/models_comparison_bar.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"💾 Saved: {OUTPUT_DIR}/models_comparison_bar.png")

### 7.2 Heatmap

In [ ]:
# Create heatmap for key metrics
fig, ax = plt.subplots(figsize=(10, 6))

heatmap_data = results_df[key_metrics]

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.4f',
    cmap='YlGnBu',
    cbar_kws={'label': 'Score'},
    linewidths=0.5,
    ax=ax,
    vmin=0,
    vmax=1
)

ax.set_title('Segmentation Models Metrics Heatmap', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Models', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/models_comparison_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"💾 Saved: {OUTPUT_DIR}/models_comparison_heatmap.png")

### 7.3 Radar Chart for Best Model

In [ ]:
# Find overall best model (based on Dice coefficient)
if 'dice_coefficient' in results_df.columns:
    best_model_name = results_df['dice_coefficient'].idxmax()
else:
    best_model_name = results_df.index[0]

print(f"🏆 Best Model: {best_model_name}")

# Prepare data for radar chart
categories = key_metrics
values = [results_df.loc[best_model_name, m] for m in categories]

# Number of variables
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values += values[:1]  # Complete the circle
angles += angles[:1]

# Create radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

ax.plot(angles, values, 'o-', linewidth=2, label=best_model_name, color='#2E86AB')
ax.fill(angles, values, alpha=0.25, color='#2E86AB')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8)
ax.grid(True, linestyle='--', alpha=0.7)

ax.set_title(f'{best_model_name} - Performance Radar', 
             size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/best_model_radar.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"💾 Saved: {OUTPUT_DIR}/best_model_radar.png")

## 8. Sample Predictions Comparison

In [ ]:
# Get sample batch
for images, masks in test_ds.take(1):
    sample_images = images[:3]  # Take 3 samples
    sample_masks = masks[:3]
    break

# Create comparison visualization
n_models = len(AVAILABLE_MODELS)
n_samples = len(sample_images)

fig, axes = plt.subplots(n_samples, n_models + 2, figsize=(4*(n_models+2), 4*n_samples))

if n_samples == 1:
    axes = axes.reshape(1, -1)

for row in range(n_samples):
    # Original image
    axes[row, 0].imshow(sample_images[row].numpy())
    axes[row, 0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[row, 0].axis('off')
    
    # Ground truth
    axes[row, 1].imshow(sample_masks[row].numpy()[:, :, 0], cmap='hot')
    axes[row, 1].set_title('Ground Truth', fontsize=12, fontweight='bold')
    axes[row, 1].axis('off')
    
    # Model predictions
    for col, (model_name, model_path) in enumerate(AVAILABLE_MODELS.items(), start=2):
        model = tf.keras.models.load_model(model_path)
        pred = model.predict(sample_images[row:row+1], verbose=0)
        
        axes[row, col].imshow(pred[0][:, :, 0], cmap='hot')
        axes[row, col].set_title(model_name, fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

plt.suptitle('Model Predictions Comparison', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/predictions_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"💾 Saved: {OUTPUT_DIR}/predictions_comparison.png")

## 9. Summary Report

In [ ]:
print("\n" + "="*60)
print("🎯 EVALUATION SUMMARY")
print("="*60)

print(f"\nModels Evaluated: {len(AVAILABLE_MODELS)}")
for i, model_name in enumerate(AVAILABLE_MODELS.keys(), 1):
    print(f"  {i}. {model_name}")

print(f"\nDataset:")
print(f"  Images Directory: {TEST_IMAGES_DIR}")
print(f"  Masks Directory: {TEST_MASKS_DIR}")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")

print(f"\n🏆 Best Performing Model:")
print(f"  Model: {best_model_name}")
for metric in key_metrics:
    value = results_df.loc[best_model_name, metric]
    print(f"  {metric}: {value:.4f}")

print(f"\nOutput Files:")
print(f"  📊 CSV: {csv_path}")
print(f"  📈 Bar Chart: {OUTPUT_DIR}/models_comparison_bar.png")
print(f"  🔥 Heatmap: {OUTPUT_DIR}/models_comparison_heatmap.png")
print(f"  🎯 Radar Chart: {OUTPUT_DIR}/best_model_radar.png")
print(f"  🖼️ Predictions: {OUTPUT_DIR}/predictions_comparison.png")

print("\n" + "="*60)
print("✅ Evaluation Complete!")
print("="*60)